In [ ]:
import sys, os, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "flask", "flask-cors", "pyngrok", "openai", "requests", "-q"])
for url in [
    "https://raw.githubusercontent.com/omar-florez/agent-coliseum/bcp/agent_base.py",
    "https://raw.githubusercontent.com/omar-florez/agent-coliseum/bcp/agent_server.py",
]:
    subprocess.check_call(["wget", "-q", "-N", url])
os.environ["AZURE_API_KEY"]  = "YOUR_AZURE_API_KEY_HERE"
os.environ["AZURE_BASE_URL"] = "https://rsgd15-foundry.openai.azure.com/openai/v1/"
os.environ["NGROK_TOKEN"]    = "YOUR_NGROK_TOKEN_HERE"
print(f"Python: {sys.executable}")
print("Setup complete")

In [ ]:
# ============================================================
#  AGENT COLISEUM — BCP Branch — Colab 02: LangChain Agent
# ============================================================
#
#  Before running:
#    1. Click the 🔑 icon in the Colab left sidebar (Secrets)
#    2. Add these secrets:
#         AZURE_API_KEY   → key provided by organizer
#         AZURE_BASE_URL  → https://rsgd15-foundry.openai.azure.com/openai/v1/
#         NGROK_TOKEN     → your ngrok token (ngrok.com)
#    3. Run all cells in order
# ============================================================
#
#  Strategy: LangChain agent with structured memory.
#    - Uses plain list to track match history (no external memory lib)
#    - Uses RunnableSequence (LCEL) for CoT
#    - No external tools during the match — fast responses
#
#  Demonstrates how LangChain abstractions map to the arena API.
# ============================================================

# ── CELL 1: Install ──────────────────────────────────────────
# !pip install flask flask-cors pyngrok langchain langchain-openai \
#              langchain-community requests -q

# ── CELL 2: Config ───────────────────────────────────────────
import os, json, random
from agent_base import Agent, MatchContext, MatchResult, WorldContext, Position
from agent_server import serve_and_register

import os
AZURE_API_KEY  = os.environ.get('AZURE_API_KEY',  'YOUR_AZURE_API_KEY_HERE')
AZURE_BASE_URL = os.environ.get('AZURE_BASE_URL', 'https://rsgd15-foundry.openai.azure.com/openai/v1/')
MODEL          = 'gpt-5'
ARENA_URL      = 'https://agent-coliseum.onrender.com'
NGROK_TOKEN    = os.environ.get('NGROK_TOKEN', 'your_ngrok_token')

# Set env vars so LangChain picks them up automatically
os.environ['OPENAI_API_KEY']  = AZURE_API_KEY
os.environ['OPENAI_BASE_URL'] = AZURE_BASE_URL

# ── CELL 4: LangChain-style chain setup (using openai directly) ───────────
# Note: replaces LangChain imports with plain openai client to avoid
# transformers/importlib-metadata conflicts in Azure ML environments.
# The chain pattern (prompt → llm → output parser) is preserved in logic.

SYSTEM_PROMPT = """You are a competitive Latin America knowledge agent.
You think carefully before asking or answering.
Use this 6-step reasoning structure:
SITUATION / OPPONENT / GOAL / DRAFT / CRITIQUE / FINAL
Each step on its own line. FINAL contains only the question or answer (1-2 sentences max)."""

def think_chain_invoke(input_text: str) -> str:
    """Mimics LangChain LCEL: prompt | llm | StrOutputParser."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": input_text},
        ],
        max_tokens=250,
    )
    return resp.choices[0].message.content.strip()

# ── CELL 5: Agent implementation ─────────────────────────────

class LangChainLatAmAgent(Agent):
    """
    LangChain-powered agent.
    Uses LCEL chain for structured thinking.
    Maintains per-match history as a plain list.
    """

    name        = "LangChain Puma"
    avatar      = "🐆"
    description = "Agente construido con LangChain LCEL y memoria de conversacion"

    def __init__(self):
        self._match_memory   = {}  # match_id → list of {turn, role, summary}
        self._opponent_notes = {}  # opponent_id → {name, result, topic}

    # ── lifecycle ─────────────────────────────────────────────

    def on_arena_start(self, ctx: WorldContext) -> None:
        self._match_memory   = {}
        self._opponent_notes = {}
        print(f"[{self.name}] Arena started with {len(ctx.agents)} agents.")

    def on_match_start(self, ctx: MatchContext) -> None:
        self._match_memory[ctx.match_id] = []
        print(f"[{self.name}] Starting match {ctx.match_id} vs {ctx.opponent_name}")

    def on_match_end(self, ctx: MatchContext, result: MatchResult) -> None:
        won = result.winner_id == ctx.my_agent_id
        self._opponent_notes[ctx.opponent_agent_id] = {
            "name":   ctx.opponent_name,
            "result": "won" if won else "lost",
            "topic":  ctx.topic,
        }
        print(f"[{self.name}] Match over: {'WON' if won else 'LOST'}")

    # ── world ─────────────────────────────────────────────────

    def move(self, ctx: WorldContext) -> Position:
        dx, dy = random.choice([(0,1),(0,-1),(1,0),(-1,0),(0,0)])
        return Position(
            x=max(0, min(ctx.map_width  - 1, ctx.my_position.x + dx)),
            y=max(0, min(ctx.map_height - 1, ctx.my_position.y + dy)),
        )

    # ── cognition ─────────────────────────────────────────────

    def think(self, ctx: MatchContext) -> str:
        """
        Calls LangChain LCEL chain with structured prompt.
        Uses plain list memory to include conversation history.
        """
        # Build history summary from match history
        history = ""
        for t in ctx.history[-3:]:
            history += (
                f"\nTurn {t['turn_number']}: "
                f"Q={t['question'][:50]} "
                f"A={t['answer'][:50]} "
                f"Score={t['score']}"
            )

        # Local memory summary (what this agent remembers from this match)
        local_mem = self._match_memory.get(ctx.match_id, [])
        mem_summary = ""
        for entry in local_mem[-2:]:
            mem_summary += f"\n  Turn {entry['turn']} ({entry['role']}): {entry['summary']}"

        # Opponent notes from past matches
        opp = self._opponent_notes.get(ctx.opponent_agent_id, {})
        opp_text = (
            f"Previously faced them: {opp.get('result','unknown')} on topic {opp.get('topic','?')}"
            if opp else "First time meeting this opponent."
        )

        input_text = f"""CURRENT MATCH:
Topic: {ctx.topic}
My role: {ctx.role}  |  Turn {ctx.turn}/{ctx.total_turns}
My score: {sum(ctx.my_scores)} pts  |  Opponent: {sum(ctx.opponent_scores)} pts
Opponent: {ctx.opponent_name}
{f'Question to answer: {ctx.current_question}' if ctx.role == 'answerer' else ''}

OPPONENT NOTES: {opp_text}

MATCH HISTORY:{history if history else ' (first turn)'}

MY LOCAL MEMORY:{mem_summary if mem_summary else ' (empty)'}

Think step by step using this reasoning structure:

# SITUATION  [Chain-of-Thought — Wei et al. 2022, NeurIPS]
# Decompose the current state before acting.
# "chain of thought prompting significantly improves the ability
#  of large language models to perform complex reasoning."
# → State the topic, your role, turn number, and current scores.

SITUATION: <assess topic, role, turn, score gap>

# OPPONENT  [Theory of Mind / Opponent Modeling — Langner et al. 2023]
# Model what your opponent knows and how they reason.
# ToM-enabled agents outperform reactive agents in competitive settings
# by anticipating the opponent's next move before it happens.
# → What are their known weaknesses? What topics did they struggle with?

OPPONENT: <model their knowledge gaps and tendencies>

# GOAL  [ReAct — Yao et al. 2022, ICLR 2023]
# Interleave reasoning with goal-directed action planning.
# "ReAct generates verbal reasoning traces and task-specific actions
#  in an interleaved manner."
# → State your concrete objective for this specific turn.

GOAL: <one concrete objective for this turn>

# DRAFT  [Scratchpad — Nye et al. 2021]
# Use an intermediate scratchpad to produce a first attempt
# before committing to a final answer.
# "Scratchpads allow models to show their work, dramatically
#  improving accuracy on multi-step reasoning tasks."
# → Write your first attempt at the question or answer.

DRAFT: <first attempt — question if asker, answer if answerer>

# CRITIQUE  [Self-Refine — Madaan et al. 2023, NeurIPS]
# Iteratively improve output using self-generated feedback.
# "Self-Refine produces significantly better outputs than
#  one-step generation across diverse tasks."
# → Is the draft good enough? Too vague? Missing a key fact?

CRITIQUE: <evaluate draft quality, identify gaps>

# FINAL  [Reflexion — Shinn et al. 2023]
# Commit to a revised response informed by self-reflection.
# "Reflexion agents verbally reflect on task feedback signals
#  to maintain an episodic memory buffer."
# → Write the final polished question (1 sentence) or answer (1-2 sentences).

FINAL: <final question (1 sentence) or answer (1-2 sentences max, be concise)>
"""

        result = think_chain_invoke(input_text)

        # Store a summary in local memory
        mem = self._match_memory.get(ctx.match_id, [])
        mem.append({
            "turn":    ctx.turn,
            "role":    ctx.role,
            "summary": result[:150],
        })
        self._match_memory[ctx.match_id] = mem

        return result

    def ask(self, ctx: MatchContext) -> str:
        scratchpad = self.think(ctx)
        return self._extract_final(scratchpad)

    def answer(self, ctx: MatchContext) -> str:
        scratchpad = self.think(ctx)
        return self._extract_final(scratchpad)

    def _extract_final(self, text: str) -> str:
        """
        Extract the FINAL answer from the reasoning trace.
        Robust to different GPT formatting styles:
          - "FINAL: answer here"
          - "**FINAL:** answer here"
          - "FINAL:\nanswer on next line"
          - Falls back to last non-empty line
        """
        lines = text.split("\n")
        for i, line in enumerate(lines):
            clean = line.replace("**", "").strip()
            if clean.upper().startswith("FINAL:"):
                rest = clean[6:].strip()
                if rest:
                    return rest
                for j in range(i+1, len(lines)):
                    if lines[j].strip():
                        return lines[j].strip()
            elif clean.upper() == "FINAL":
                for j in range(i+1, len(lines)):
                    if lines[j].strip():
                        return lines[j].strip()
        for line in reversed(lines):
            if line.strip() and not line.strip().startswith("#"):
                return line.strip()
        return text.strip()


# ── CELL 5: Run ──────────────────────────────────────────────
# ── Keepalive — prevents ngrok tunnel from dropping ───────────────────────
import threading, requests, time

def _keepalive(arena_url):
    """Ping the arena every 60s to keep the ngrok tunnel and Colab session alive."""
    while True:
        time.sleep(60)
        try:
            requests.get(f"{arena_url}/health", timeout=5)
        except:
            pass

threading.Thread(target=_keepalive, args=(ARENA_URL,), daemon=True).start()
print("Keepalive thread started")

from pyngrok import ngrok
ngrok.kill()  # kill any existing tunnels before starting

agent = LangChainLatAmAgent()

serve_and_register(
    agent       = agent,
    arena_url   = ARENA_URL,
    port        = 5001,
    ngrok_token = NGROK_TOKEN,
)
# This cell blocks. The agent is now live and registered.
# Wait for the organizer to accept you in the admin panel.
